# Soluciones — Clase 18: Feature engineering — crear mejores variables

Este notebook contiene soluciones de referencia para los ejercicios de la clase.
Intenta resolverlos por tu cuenta antes de consultar estas soluciones.

> El feature engineering no tiene una sola respuesta correcta.
> Dos analistas pueden preparar el mismo dataset de formas diferentes
> y ambas ser válidas según el contexto.

In [ ]:
# Configuración inicial
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler

sns.set_theme(style="whitegrid", palette="muted")

ventas = pd.read_csv("../../datasets/ventas_tienda.csv")
estudiantes = pd.read_csv("../../datasets/estudiantes.csv")

print("Datos cargados.")
print("Ventas:", ventas.shape, "| Estudiantes:", estudiantes.shape)

In [ ]:
# SOLUCIÓN — Ejercicio 1: Exploración inicial

print("=== TIPOS DE DATOS ===")
print(ventas.dtypes)
print()

print("=== VALORES NULOS ===")
nulos = ventas.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin valores nulos.")
print()

print("=== ESTADÍSTICAS DESCRIPTIVAS ===")
print(ventas.describe().round(2))
print()

# Análisis de candidatos para features
print("=== CANDIDATOS PARA NUEVAS VARIABLES ===")
print("Numéricas derivables:", ventas.select_dtypes(include="number").columns.tolist())
print("Categóricas para encoding:", ventas.select_dtypes(include="object").columns.tolist())

In [ ]:
# SOLUCIÓN — Ejercicio 2: Variables numéricas derivadas

df = ventas.copy()

# Variable 1: ingreso bruto (sin descuento)
df["ingreso_bruto"] = df["precio_unitario"] * df["unidades"]

# Variable 2: ingreso neto (después del descuento)
df["ingreso_neto"] = df["ingreso_bruto"] * (1 - df["descuento"] / 100)

# Variable 3: ganancia por unidad
df["ganancia_por_unidad"] = df["precio_unitario"] - df["costo_unitario"]

# Variable 4: ratio descuento (fracción del precio descontada)
# Protección contra división por cero
df["ratio_descuento"] = np.where(
    df["precio_unitario"] > 0,
    df["descuento"] / df["precio_unitario"],
    0
)

print("Variables derivadas creadas:")
nuevas_cols = ["precio_unitario", "unidades", "descuento", "ingreso_bruto", "ingreso_neto", "ganancia_por_unidad", "ratio_descuento"]
print(df[nuevas_cols].head(5).round(2))

In [ ]:
# SOLUCIÓN — Ejercicio 3: Extracción desde fechas

df["fecha"] = pd.to_datetime(df["fecha"])

df["mes_venta"]      = df["fecha"].dt.month
df["dia_semana_num"] = df["fecha"].dt.dayofweek
nombres_dias = {0: "Lunes", 1: "Martes", 2: "Miércoles",
                3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}
df["nombre_dia"]     = df["dia_semana_num"].map(nombres_dias)
df["es_fin_semana"]  = (df["dia_semana_num"] >= 5).astype(int)
df["trimestre"]      = df["fecha"].dt.quarter

print("Componentes temporales extraídas:")
print(df[["fecha", "mes_venta", "dia_semana_num", "nombre_dia", "es_fin_semana", "trimestre"]].head(8))
print()
print(f"Ventas en fin de semana: {df['es_fin_semana'].sum()} ({df['es_fin_semana'].mean()*100:.1f}%)")
print(f"Ventas días de semana:   {(df['es_fin_semana'] == 0).sum()}")

In [ ]:
# SOLUCIÓN — Ejercicio 4: One-hot encoding

print("Categorías antes del encoding:", df["categoria"].unique())
print("Cantidad de categorías:", df["categoria"].nunique())
print()

# Sin drop_first: genera una columna por cada categoría
dummies_completas = pd.get_dummies(df["categoria"], prefix="cat")
print(f"Columnas sin drop_first: {dummies_completas.shape[1]}")
print(dummies_completas.columns.tolist())
print()

# Con drop_first=True: elimina la primera columna para evitar multicolinealidad perfecta
# (si todas las demás son 0, sabemos que corresponde a la categoría eliminada)
dummies_reducidas = pd.get_dummies(df["categoria"], prefix="cat", drop_first=True)
print(f"Columnas con drop_first=True: {dummies_reducidas.shape[1]}")
print(dummies_reducidas.columns.tolist())
print()
print("Se usa drop_first=True para evitar la trampa de la variable dummy (multicolinealidad perfecta).")

df = pd.concat([df, dummies_reducidas], axis=1)

In [ ]:
# SOLUCIÓN — Ejercicio 5: Codificación ordinal

# Crear la columna 'nivel_estudio' manualmente si no existe
if "nivel_estudio" not in estudiantes.columns:
    np.random.seed(42)
    estudiantes["nivel_estudio"] = np.random.choice(
        ["primaria", "secundaria", "universitario"],
        size=len(estudiantes)
    )
    print("Columna 'nivel_estudio' creada manualmente para el ejercicio.")

# Diccionario de mapeo ordinal
mapa_nivel = {
    "primaria":      0,   # nivel más bajo
    "secundaria":    1,   # nivel medio
    "universitario": 2    # nivel más alto
}

estudiantes["nivel_cod"] = estudiantes["nivel_estudio"].map(mapa_nivel)

print("\nCodificación ordinal:")
print(estudiantes[["nivel_estudio", "nivel_cod"]].drop_duplicates().sort_values("nivel_cod"))

In [ ]:
# SOLUCIÓN — Ejercicio 6: Binning con pd.cut y pd.qcut

# pd.cut: rangos definidos manualmente
estudiantes["grupo_nota_cut"] = pd.cut(
    estudiantes["nota_final"],
    bins=[0, 49, 69, 89, 100],
    labels=["reprobado", "suficiente", "bueno", "excelente"],
    include_lowest=True
)

# pd.qcut: rangos de igual cantidad de observaciones
estudiantes["grupo_nota_qcut"] = pd.qcut(
    estudiantes["nota_final"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

print("=== pd.cut (rangos manuales) ===")
print(estudiantes["grupo_nota_cut"].value_counts())
print()
print("=== pd.qcut (cuartiles) ===")
print(estudiantes["grupo_nota_qcut"].value_counts().sort_index())
print()
print("Diferencia: pd.cut puede crear grupos desiguales en tamaño si los datos no son uniformes.")
print("pd.qcut garantiza grupos con la misma cantidad de registros.")

In [ ]:
# SOLUCIÓN — Ejercicios 7 y 8: Escalado + Selección por correlación

cols_escalar = ["precio_unitario", "unidades", "ingreso_neto"]
datos_num = df[cols_escalar].fillna(0)

# StandardScaler
scaler_std = StandardScaler()
std_data = pd.DataFrame(scaler_std.fit_transform(datos_num),
                        columns=[c + "_std" for c in cols_escalar])

# MinMaxScaler
scaler_mm = MinMaxScaler()
mm_data = pd.DataFrame(scaler_mm.fit_transform(datos_num),
                       columns=[c + "_mm" for c in cols_escalar])

print("StandardScaler — media ≈ 0, std ≈ 1:")
print(std_data.agg(["mean", "std"]).round(3))
print()
print("MinMaxScaler — rango [0, 1]:")
print(mm_data.agg(["min", "max"]).round(3))
print()

# Selección por correlación con el target
df["venta_alta"] = (df["ingreso_neto"] > df["ingreso_neto"].median()).astype(int)
corr_target = df.select_dtypes(include="number").corr()["venta_alta"].drop("venta_alta")
corr_target = corr_target.sort_values(key=abs, ascending=False)

print("=== TOP 10 VARIABLES POR CORRELACIÓN CON 'venta_alta' ===")
print(corr_target.head(10).round(3))
print()

vars_utiles = corr_target[corr_target.abs() > 0.05].index.tolist()
print(f"Variables seleccionadas (correlación > 0.05): {len(vars_utiles)}")

## Reflexión final

El feature engineering es una habilidad que se desarrolla con práctica y conocimiento del dominio.
No hay una receta única — cada dataset y cada problema requieren decisiones distintas.

Lo que sí es universal:
- Explorar antes de transformar.
- Documentar cada decisión.
- Verificar que los valores derivados tengan sentido antes de usarlos.
- Escalar solo en el conjunto de entrenamiento, nunca en el de prueba antes de tiempo.

Con las variables bien preparadas, estás listo para los modelos de machine learning.
Ese es el siguiente capítulo del bootcamp.